# Treinando modelo

Abrindo base de dados

In [ ]:
import pandas

data_frame = pandas.read_csv('Dataset/processed/plantvillage_data.csv')
data_frame

,Unnamed: 0,file_path,version,class,plant,disease
0,0,Dataset/plantvillage_dataset/segmented/Apple__...,segmented,Apple___Apple_scab,Apple,Apple_scab
1,1,Dataset/plantvillage_dataset/segmented/Apple__...,segmented,Apple___Apple_scab,Apple,Apple_scab
2,2,Dataset/plantvillage_dataset/segmented/Apple__...,segmented,Apple___Apple_scab,Apple,Apple_scab
3,3,Dataset/plantvillage_dataset/segmented/Apple__...,segmented,Apple___Apple_scab,Apple,Apple_scab
4,4,Dataset/plantvillage_dataset/segmented/Apple__...,segmented,Apple___Apple_scab,Apple,Apple_scab
...,...,...,...,...,...,...
162911,162911,Dataset/plantvillage_dataset/color/Tomato___he...,color,Tomato___healthy,Tomato,healthy
162912,162912,Dataset/plantvillage_dataset/color/Tomato___he...,color,Tomato___healthy,Tomato,healthy
162913,162913,Dataset/plantvillage_dataset/color/Tomato___he...,color,Tomato___healthy,Tomato,healthy
162914,162914,Dataset/plantvillage_dataset/color/Tomato___he...,color,Tomato___healthy,Tomato,healthy


Definindo função para o pré-processamento de imagens

In [2]:
import numpy
from PIL import Image
import tensorflow

def preprocesar_imagem(path: str):
    """
    Realiza o pré-processamento da imagem
    :param path: Caminho do arquivo de imagem
    :return: Imagem pré-processada
    """

    # Carregando imagem e redimensionando para o tamanho esperado pelo ResNet50
    imagem = numpy.array(Image.open(path).convert('RGB').resize((224, 224)))

    # Pré-Processando a imagem de acordo com o ResNet50
    imagem = tensorflow.keras.applications.resnet50.preprocess_input(imagem)

    return imagem

I0000 00:00:1781814445.176249   61514 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781814445.176570   61514 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781814445.204857   61514 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781814445.954845   61514 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENAB

Separando os dados de X e Y

In [3]:
X = data_frame['file_path']
Y = data_frame['class']

Separando os dados de teste e treino

In [4]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=1024)

Criando modelo

In [5]:
from keras.src.applications.resnet import ResNet50

base_model = ResNet50(weights='imagenet', include_top=False)

E0000 00:00:1781814446.439645   61514 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Adicionando camadas personalizadas

In [6]:
from keras.src.layers import Dense
from keras.src.layers.pooling.global_average_pooling2d import GlobalAveragePooling2D

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(len(data_frame['class'].unique()), activation='softmax')(x)

Definindo modelo completo

In [7]:
from keras import Model

model = Model(inputs=base_model.input, outputs=predictions)

Congelando as camadas do modelo base

In [8]:
for layer in base_model.layers:
    layer.trainable = False

Compilando o modelo

In [9]:
from keras.src.optimizers import Adam

model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

Treinando o modelo ResNet50

In [10]:
for caminho_imagem in x_train:
    imagem = preprocesar_imagem(caminho_imagem)
    model.fit(imagem)

ValueError: Exception encountered when calling Functional.call().

[1mInvalid input shape for input Tensor("data:0", shape=(32, 224, 3), dtype=float32) with name 'keras_tensor' and path ''. Expected shape (None, None, None, 3), but input has incompatible shape (32, 224, 3)[0m

Arguments received by Functional.call():
  • inputs=tf.Tensor(shape=(32, 224, 3), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>